In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


## First Try no wrapper 

In [2]:
from copy import deepcopy

from enviroment_bj import BlackjackEnvironment, BlackjackConfig, ObservationConfig, StartStateConfig
from model.encoder import EncoderConfig, BlackjackObservationEncoder
from model.agents import FeedForwardDoubleDQN, AgentNetworkConfig

from loss import BellmanLossConfig, LossPhaseWeightConfig

from training import (
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    DualEpsilonConfig,
    NStepConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
    TransferLearningConfig,
    DistillationConfig,
    train_model,
)


In [3]:
TRANSFER_EXPERIMENT = {
    # =========================
    # CHECKPOINTS BASE
    # =========================
    "teacher_checkpoint_path": r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\baseline_playing_feedforward_v1\best_eval.pt",
    "warm_start_checkpoint_path": r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\baseline_playing_feedforward_v1\best_eval.pt",

    # =========================
    # STAGE
    # =========================
    "stage_name": "stage_01_realistic_lite_fresh_from_baseline_v1",
    "notes": (
        "Transfer desde baseline minimal_basic_strategy hacia realista-lite fresh_shoe. "
        "FeedForwardDoubleDQN, bet_1x, sin temporal/history/discard, con distillation q_mse."
    ),

    # =========================
    # TRANSFER
    # =========================
    "use_resume": False,
    "use_warm_start": True,
    "use_teacher": True,

    # =========================
    # DISTILLATION
    # =========================
    "distillation_enabled": True,
    "distillation_mode": "q_mse",
    "distillation_weight": 0.25,
    "distillation_final_weight": 0.05,
    "distillation_decay_steps": 50_000,
    "distillation_temperature": 1.0,
    "distillation_playing_only": True,
}

In [4]:
# ============================================================
# OBSERVATION: realista-lite
# ------------------------------------------------------------
# Objetivo:
# - Mantener table_raw.
# - Mantener reglas, hand_context, other_hands.
# - Quitar temporal/history/discard por ahora.
# ============================================================

observation_config = ObservationConfig(
    profile="table_realistic_unknown_progress",
    obs_include_table_rules=True,
    obs_include_visible_rules_only=True,
    obs_include_hidden_rules=False,

    obs_include_decision_phase=True,
    obs_include_available_bet_multipliers=True,

    obs_current_hand_mode="table_raw",
    obs_include_other_player_hands=True,
    obs_include_current_bet=True,
    obs_include_betting_context=True,
    obs_include_hand_context=True,
    obs_include_insurance_context=True,

    # Apagados en realista-lite
    obs_include_temporal_context=False,
    obs_include_hands_since_shuffle=False,
    obs_include_estimated_shoe_progress=False,
    obs_include_last_hand_outcome=False,
    obs_include_recent_actions=False,

    # Apagados en realista-lite
    obs_include_observed_cards_history=False,
    obs_observed_cards_mode="rank_counts",
    obs_recent_cards_window=32,
    obs_reset_history_on_shuffle=True,

    obs_include_exact_shoe_composition=False,
    obs_include_discard_summary=False,

    obs_include_n_decks=False,
    obs_include_shoe_penetration_rule=False,
)

In [5]:
# ============================================================
# START STATE
# ------------------------------------------------------------
# Fresh shoe para aislar si el problema es la representación
# realista o el unknown_progress.
# ============================================================

start_state_config = StartStateConfig(
    mode="fresh_shoe",
    min_burned_rounds=0,
    max_burned_rounds=0,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=False,
)

In [6]:
# ============================================================
# TABLE CONFIG
# ------------------------------------------------------------
# Realista-lite:
# - 8 decks.
# - S17.
# - 3:2.
# - bet_1x solamente.
# - sin exogenous cards.
# - sin insurance por ahora.
# - sin surrender por ahora.
# - sin six-card charlie.
# ============================================================

blackjack_config = BlackjackConfig(
    n_decks=8,
    shoe_penetration=0.75,
    use_cut_card=True,
    visible_shoe_change=True,

    exogenous_cards=False,
    simulate_exogenous_visible_cards=False,
    exogenous_visible_cards_mode="disabled",

    dealer_hits_soft_17=False,
    blackjack_payout=1.5,
    dealer_peeks_for_blackjack=True,

    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    double_split_aces_allowed=False,

    split_rule="same_value",
    max_hands_after_split=2,
    max_split_depth_per_hand=1,
    resplit_aces_allowed=False,
    hit_split_aces_allowed=False,

    surrender_allowed=False,
    insurance_allowed=False,
    six_card_charlie_enabled=False,

    base_bet=1.0,
    bet_multipliers=(1,),
    strict_shoe_validation=False,

    observation=observation_config,
    observation_mode=None,
    expose_shoe_composition=False,
)

In [8]:
# ============================================================
# ENVS
# ============================================================

base_seed = 1241651

envs = [
    BlackjackEnvironment(
        config=deepcopy(blackjack_config),
        seed=base_seed,
        start_state=start_state_config,
    )
]

In [11]:
# ============================================================
# ENCODER: realista-lite
# ============================================================

encoder_config = EncoderConfig(
    profile="table_realistic_unknown_progress",

    encode_rules=True,
    encode_betting_context=True,
    encode_other_hands=True,

    # Apagados en esta etapa
    encode_temporal=False,
    encode_observed_history=False,
    encode_discard_summary=False,
    encode_recent_actions=False,

    encode_exact_shoe=False,
    encode_action_mask_features=False,

    card_encoding="one_hot_rank",
    history_encoding="rank_counts",
    normalize_counts=True,
    use_visible_table_rules_only=True,

    max_current_hand_cards=12,
    max_cards_per_hand=12,
    max_other_hands=4,
    max_recent_actions=5,
    max_recent_cards=32,
    max_recent_discard_cards=16,
)

encoder = BlackjackObservationEncoder(config=encoder_config)

# ============================================================
# MODEL: student feedforward realista-lite
# ============================================================

model_config = AgentNetworkConfig.for_architecture(
    architecture="feedforward",
    encoder_profile=encoder_config.profile,
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    feedforward_hidden_dims=(256, 256, 128),
    use_phase_adapters=False,
    use_module_gating=False,
)

model = FeedForwardDoubleDQN(
    config=model_config,
    encoder=encoder,
)


# ============================================================
# LOSS
# ============================================================

loss_config = BellmanLossConfig(
    gamma=0.99,
    loss_type="huber",
    validate_current_actions=True,
    validate_next_action_mask=True,
    allow_terminal_without_legal_next_action=True,
    phase_weights=LossPhaseWeightConfig(
        enabled=True,
        betting_weight=0.25,
        playing_weight=1.50,
    ),
)

# ============================================================
# EPSILON
# ============================================================

dual_epsilon_config = DualEpsilonConfig(
    betting=EpsilonScheduleConfig(
        start=0.20,
        end=0.02,
        decay_steps=40_000,
        evaluation_epsilon=0.0,
    ),
    playing=EpsilonScheduleConfig(
        start=0.18,
        end=0.05,
        decay_steps=80_000,
        evaluation_epsilon=0.0,
    ),
)

# ============================================================
# N-STEP
# ============================================================

n_step_config = NStepConfig(
    enabled=True,
    n_steps=3,
)

# ============================================================
# REPLAY BUFFER
# ============================================================

replay_buffer_config = ReplayBufferConfig(
    capacity=120_000,
    batch_size=128,
    warmup_size=12_000,
    sequence_length=8,
    min_sequence_length=2,
)

# ============================================================
# OPTIMIZATION
# ============================================================

optimization_config = OptimizationConfig(
    optimizer="adamw",
    learning_rate=1e-4,
    weight_decay=1e-5,
    scheduler="step",
    scheduler_step_size=25_000,
    scheduler_gamma=0.97,
    gradient_clipping=True,
    max_grad_norm=5.0,
)

# ============================================================
# TARGET UPDATE
# ============================================================

target_update_config = TargetUpdateConfig(
    mode="soft",
    hard_update_interval=1000,
    soft_tau=0.005,
)

# ============================================================
# EVALUATION
# ============================================================

evaluation_config = EvaluationConfig(
    enabled=True,
    every_n_epochs=1,
    num_rounds=2500,
    max_decisions=25_000)


# ============================================================
# CHECKPOINTS
# ============================================================

checkpoint_config = CheckpointConfig(
    directory=fr"training_checkpoints\{TRANSFER_EXPERIMENT['stage_name']}",
    save_latest=True,
    save_best_eval=True,
    save_periodic=True,
    periodic_interval_updates=2500,
    best_metric_name="ev_per_1000_hands",
    maximize_best_metric=True,
)

# ============================================================
# PRINTS
# ============================================================

print_config = PrintConfig(
    enable=True,
    print_run_summary=True,
    print_warmup_interval=1000,
    print_update_interval=200,
    print_collection_interval=1000,
    print_epoch_header=True,
    print_epoch_summary=True,
    print_eval_summary=True,
    include_segment_details=False,
)




In [13]:
# ============================================================
# TRAINER
# ============================================================

trainer_config = TrainerConfig(
    total_epochs=25,
    env_steps_per_epoch=4500,
    train_frequency=4,
    updates_per_train_step=1,
    max_updates_per_epoch=None,
    device="cpu",  # cambia a "cuda" si lo tienes disponible
    seed=44,
    reset_hidden_on_round_end=False,
    sequence_end_on_done=False,
    flush_partial_sequences_at_epoch_end=True,
    loss=loss_config,
)

# ============================================================
# TRANSFER LEARNING CONFIG
# ------------------------------------------------------------
# Warm-start + teacher distillation.
# NO usar resume=True.
# ============================================================

transfer_config = TransferLearningConfig(
    enabled=True,
    warm_start_checkpoint_path=TRANSFER_EXPERIMENT["warm_start_checkpoint_path"],
    teacher_checkpoint_path=TRANSFER_EXPERIMENT["teacher_checkpoint_path"],
    distillation=DistillationConfig(
        enabled=TRANSFER_EXPERIMENT["distillation_enabled"],
        mode=TRANSFER_EXPERIMENT["distillation_mode"],
        weight=TRANSFER_EXPERIMENT["distillation_weight"],
        final_weight=TRANSFER_EXPERIMENT["distillation_final_weight"],
        decay_steps=TRANSFER_EXPERIMENT["distillation_decay_steps"],
        temperature=TRANSFER_EXPERIMENT["distillation_temperature"],
        playing_only=TRANSFER_EXPERIMENT["distillation_playing_only"],
    ),
)


pipeline_config = TrainingPipelineConfig(
    trainer=trainer_config,
    replay_buffer=replay_buffer_config,
    epsilon=dual_epsilon_config,
    n_step=n_step_config,
    optimization=optimization_config,
    target_update=target_update_config,
    evaluation=evaluation_config,
    checkpoints=checkpoint_config,
    transfer=transfer_config,
    prints=print_config,
)

In [14]:
# ============================================================
# TRAIN
# ------------------------------------------------------------
# Transfer learning:
# - warm_start desde best_eval baseline
# - teacher distillation desde best_eval baseline
# - NO resume
# ============================================================

result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
    resume=False,
    resume_checkpoint_path=None,
)

print("Warm-start report:")
print(result.get("warm_start_report"))

print("Checkpoint dir:")
print(result.get("checkpoint_dir"))

BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=fresh_shoe
  Runtime    : device=cpu | epochs=25 | envs=1 | steps/epoch=4500 | updates/epoch~=1125 | params=336,785
  Optim      : optimizer=adamw | lr=1.00e-04 | loss=huber | gamma=0.9900 | grad_clip=True(5.00)
  Replay     : warmup=12000 | capacity=120000 | batch=128 | seq_len=8 | min_seq_len=2
  Explore    : eps_bet=0.200->0.020 (decay 40000) | eps_play=0.180->0.050 (decay 80000) | target=soft | interval=1000 | tau=0.0050
  Extras     : n_step=True(3) | phase_loss_w=True (bet 0.25, play 1.50) | phase_adapters=False | module_gating=False
  Transfer   : enabled=True | warm_start=C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\baseline_playing_feedforward_v1\best_eval.pt | teacher=C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackj

---

---

In [16]:
from training.final_wrapper import * 

stage_02a_result = run_blackjack_transfer_stage(
    stage_name="stage_02a_realistic_history_fresh_from_stage01_v1",

    output_root=r'C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints',

    # Warm-start desde Stage 01
    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_01_realistic_lite_fresh_from_baseline_v1\latest.pt",

    # Teacher sigue siendo el baseline original
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\baseline_playing_feedforward_v1\best_eval.pt",

    # Distillation un poco más suave
    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.20,
    distillation_final_weight=0.05,
    distillation_decay_steps=50_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    # Activamos SOLO observed_history
    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=False,
    include_recent_actions=False,

    # Todavía fresh_shoe
    start_mode="fresh_shoe",

    # Todavía sin apuestas
    bet_multipliers=(1,),

    # Mesa todavía limpia
    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    # Un solo env
    penetrations=[0.75],
    base_seed=45,

    # Modelo compatible con warm-start
    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    # Training diagnóstico
    total_epochs=25,
    env_steps_per_epoch=4500,
    device="cpu",

    # Replay
    replay_capacity=120_000,
    batch_size=128,
    warmup_size=12_000,

    # Optim
    learning_rate=1e-4)

BLACKJACK TRANSFER STAGE
stage_name: stage_02a_realistic_history_fresh_from_stage01_v1
checkpoint_dir: notebooks\training_checkpoints\stage_02a_realistic_history_fresh_from_stage01_v1
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_01_realistic_lite_fresh_from_baseline_v1\latest.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\baseline_playing_feedforward_v1\best_eval.pt
start_mode: fresh_shoe
include_observed_history: True
include_discard_summary: False
include_temporal_context: False
include_recent_actions: False
bet_multipliers: (1,)
penetrations: [0.75]
state_dim: 938
BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=fresh_shoe
  Runtime    : device=cpu | epochs=25 | envs=1 | 

---

# Discard Summary True 

In [18]:
stage_02b_result = run_blackjack_transfer_stage(
    stage_name="stage_02c_realistic_history_temporal_fresh_from_stage02a_v1",
    output_root=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints",

    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_02A_realistic_history_fresh_feedforward_best_eval.pt",
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt",

    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.15,
    distillation_final_weight=0.04,
    distillation_decay_steps=50_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=True,
    include_recent_actions=False,

    start_mode="fresh_shoe",

    bet_multipliers=(1,),
    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    penetrations=[0.75],
    base_seed=47,

    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    total_epochs=25,
    env_steps_per_epoch=4500,
    device="cpu",

    replay_capacity=120_000,
    batch_size=128,
    warmup_size=12_000,

    learning_rate=1e-4,
)

BLACKJACK TRANSFER STAGE
stage_name: stage_02c_realistic_history_temporal_fresh_from_stage02a_v1
checkpoint_dir: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_02c_realistic_history_temporal_fresh_from_stage02a_v1
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_02A_realistic_history_fresh_feedforward_best_eval.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt
start_mode: fresh_shoe
include_observed_history: True
include_discard_summary: False
include_temporal_context: True
include_recent_actions: False
bet_multipliers: (1,)
penetrations: [0.75]
state_dim: 967
BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=fresh_shoe
  

---

# Unknown Progress (mas casino like )

In [20]:
stage_03a_result = run_blackjack_transfer_stage(
    stage_name="stage_03a_unknown_progress_soft_from_stage02c_v1",
    output_root=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints",

    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_02C_realistic_history_temporal_fresh_feedforward_best_eval.pt",
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt",

    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.12,
    distillation_final_weight=0.03,
    distillation_decay_steps=50_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=True,
    include_recent_actions=False,

    start_mode="unknown_progress",
    min_burned_rounds=2,
    max_burned_rounds=10,

    bet_multipliers=(1,),
    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    penetrations=[0.75],
    base_seed=48,

    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    total_epochs=25,
    env_steps_per_epoch=4500,
    device="cpu",

    replay_capacity=120_000,
    batch_size=128,
    warmup_size=12_000,

    learning_rate=7e-5,
)

BLACKJACK TRANSFER STAGE
stage_name: stage_03a_unknown_progress_soft_from_stage02c_v1
checkpoint_dir: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_03a_unknown_progress_soft_from_stage02c_v1
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_02C_realistic_history_temporal_fresh_feedforward_best_eval.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt
start_mode: unknown_progress
include_observed_history: True
include_discard_summary: False
include_temporal_context: True
include_recent_actions: False
bet_multipliers: (1,)
penetrations: [0.75]
state_dim: 967
BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=unknown_progress
  R


--- 

## Subimos dificultad de burned 

In [21]:
stage_03b_result = run_blackjack_transfer_stage(
    stage_name="stage_03b_unknown_progress_medium_from_stage03a_v1",
    output_root=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints",

    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03A_unknown_progress_soft_feedforward_best_eval.pt",
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt",

    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.10,
    distillation_final_weight=0.02,
    distillation_decay_steps=50_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=True,
    include_recent_actions=False,

    start_mode="unknown_progress",
    min_burned_rounds=5,
    max_burned_rounds=25,

    bet_multipliers=(1,),
    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    penetrations=[0.75],
    base_seed=49,

    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    total_epochs=25,
    env_steps_per_epoch=4500,
    device="cpu",

    replay_capacity=120_000,
    batch_size=128,
    warmup_size=12_000,

    learning_rate=5e-5,
)

BLACKJACK TRANSFER STAGE
stage_name: stage_03b_unknown_progress_medium_from_stage03a_v1
checkpoint_dir: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_03b_unknown_progress_medium_from_stage03a_v1
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03A_unknown_progress_soft_feedforward_best_eval.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt
start_mode: unknown_progress
include_observed_history: True
include_discard_summary: False
include_temporal_context: True
include_recent_actions: False
bet_multipliers: (1,)
penetrations: [0.75]
state_dim: 967
BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=unknown_progress
  Runtime 

---

# Unknown con burst Hard 

In [ ]:
stage_03c_result = run_blackjack_stage(
    stage_name="stage_03c_unknown_progress_hard_from_stage03b_v1",
    output_root=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints",

    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03B_unknown_progress_medium_feedforward_best_eval.pt",
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt",

    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.08,
    distillation_final_weight=0.02,
    distillation_decay_steps=50_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=True,
    include_recent_actions=False,

    start_mode="unknown_progress",
    min_burned_rounds=10,
    max_burned_rounds=60,

    bet_multipliers=(1,),
    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    penetrations=[0.75],
    base_seed=50,

    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    total_epochs=25,
    env_steps_per_epoch=4500,
    device="cpu",

    replay_capacity=120_000,
    batch_size=128,
    warmup_size=12_000,

    learning_rate=3e-5,
)

BLACKJACK TRANSFER STAGE
stage_name: stage_03c_unknown_progress_hard_from_stage03b_v1
checkpoint_dir: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_03c_unknown_progress_hard_from_stage03b_v1
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03B_unknown_progress_medium_feedforward_best_eval.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt
start_mode: unknown_progress
include_observed_history: True
include_discard_summary: False
include_temporal_context: True
include_recent_actions: False
bet_multipliers: (1,)
penetrations: [0.75]
state_dim: 967
BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=unknown_progress
  Runtime   

---

# Abrimos betting con FeedForward a ver como nos va 

In [ ]:
import importlib
import training.final_wrapper

importlib.reload(training.final_wrapper)

from training.final_wrapper import * 


stage_04a_result = run_blackjack_stage(
    stage_name="stage_04a_betting_open_feedforward_from_stage03c_v1",
    output_root=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints",

    # ============================================================
    # TRANSFER
    # ============================================================
    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03C_unknown_progress_hard_feedforward_best_eval.pt",
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt",

    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.06,
    distillation_final_weight=0.01,
    distillation_decay_steps=50_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    # ============================================================
    # OBSERVATION / ENCODER
    # ============================================================
    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=True,
    include_recent_actions=False,

    # ============================================================
    # START STATE
    # ============================================================
    start_mode="unknown_progress",
    min_burned_rounds=10,
    max_burned_rounds=60,

    # ============================================================
    # TABLE: abrimos apuestas
    # ============================================================
    bet_multipliers=(1, 2, 3, 4),

    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    penetrations=[0.75],
    base_seed=51,

    # ============================================================
    # MODEL
    # ============================================================
    architecture="feedforward",
    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    # ============================================================
    # LOSS: ahora sí dejamos aprender betting
    # ============================================================
    betting_loss_weight=1.25,
    playing_loss_weight=0.75,

    # ============================================================
    # EPSILON
    # Betting necesita explorar bastante más que playing.
    # ============================================================
    betting_epsilon_start=0.65,
    betting_epsilon_end=0.08,
    betting_epsilon_decay_steps=120_000,
    betting_evaluation_epsilon=0.0,

    playing_epsilon_start=0.10,
    playing_epsilon_end=0.03,
    playing_epsilon_decay_steps=60_000,
    playing_evaluation_epsilon=0.0,

    # ============================================================
    # TRAINING
    # ============================================================
    total_epochs=40,
    env_steps_per_epoch=6000,
    train_frequency=4,
    updates_per_train_step=1,
    device="cpu",

    replay_capacity=200_000,
    batch_size=128,
    warmup_size=20_000,

    learning_rate=3e-5,
    weight_decay=1e-5,

    eval_rounds=3000,
    eval_max_decisions=40_000,
)

BLACKJACK HIGH-LEVEL WRAPPER
stage_name: stage_04a_betting_open_feedforward_from_stage03c_v1
architecture: feedforward
observation_profile: table_realistic_unknown_progress
encoder_profile: table_realistic_unknown_progress
start_mode: unknown_progress
num_envs: 1
penetrations: [0.75]
checkpoint_dir: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_04a_betting_open_feedforward_from_stage03c_v1
resume: False
resume_checkpoint_path: None
transfer_enabled: True
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03C_unknown_progress_hard_feedforward_best_eval.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt
distillation_enabled: True
bet_multipliers: (1, 2, 3, 4)
state_dim: 967
use_optimizer_param_groups: False
freeze_playing_parts: F

KeyboardInterrupt: 

: 

### No parecio funcionar no logra apostar asi que pasemos a algo mas extremo 

--- 

In [ ]:
import importlib
import training.final_wrapper

importlib.reload(training.final_wrapper)

from training.final_wrapper import * 


stage_04b_result = run_blackjack_stage(
    stage_name="stage_04b_betting_head_focus_from_stage03c_v1",
    output_root=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints",

    warm_start_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03C_unknown_progress_hard_feedforward_best_eval.pt",
    teacher_checkpoint_path=r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt",

    distillation_enabled=True,
    distillation_mode="q_mse",
    distillation_weight=0.04,
    distillation_final_weight=0.00,
    distillation_decay_steps=40_000,
    distillation_temperature=1.0,
    distillation_playing_only=True,

    include_observed_history=True,
    include_discard_summary=False,
    include_temporal_context=True,
    include_recent_actions=False,

    start_mode="unknown_progress",
    min_burned_rounds=10,
    max_burned_rounds=60,

    bet_multipliers=(1, 2, 3, 4),

    exogenous_cards=False,
    insurance_allowed=False,
    surrender_allowed=False,
    six_card_charlie_enabled=False,

    penetrations=[0.75],
    base_seed=52,

    architecture="feedforward",
    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,

    # ============================================================
    # Importante: proteger playing y enfocar betting
    # ============================================================
    freeze_playing_parts=True,
    use_optimizer_param_groups=True,
    backbone_lr=0.0,
    play_lr=0.0,
    bet_lr=3e-4,
    default_lr=1e-5,

    betting_loss_weight=2.0,
    playing_loss_weight=0.25,

    # ============================================================
    # Epsilon
    # Betting explora, playing casi greedy
    # ============================================================
    betting_epsilon_start=0.70,
    betting_epsilon_end=0.10,
    betting_epsilon_decay_steps=160_000,
    betting_evaluation_epsilon=0.0,

    playing_epsilon_start=0.03,
    playing_epsilon_end=0.01,
    playing_epsilon_decay_steps=40_000,
    playing_evaluation_epsilon=0.0,

    total_epochs=40,
    env_steps_per_epoch=7000,
    train_frequency=4,
    updates_per_train_step=1,
    device="cpu",

    replay_capacity=250_000,
    batch_size=128,
    warmup_size=25_000,

    learning_rate=3e-5,
    weight_decay=1e-5,

    eval_rounds=5000,
    eval_max_decisions=60_000,
)

BLACKJACK HIGH-LEVEL WRAPPER
stage_name: stage_04b_betting_head_focus_from_stage03c_v1
architecture: feedforward
observation_profile: table_realistic_unknown_progress
encoder_profile: table_realistic_unknown_progress
start_mode: unknown_progress
num_envs: 1
penetrations: [0.75]
checkpoint_dir: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\notebooks\training_checkpoints\stage_04b_betting_head_focus_from_stage03c_v1
resume: False
resume_checkpoint_path: None
transfer_enabled: True
warm_start_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\KEEP_03C_unknown_progress_hard_feedforward_best_eval.pt
teacher_checkpoint_path: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\models\Baseline_bj.pt
distillation_enabled: True
bet_multipliers: (1, 2, 3, 4)
state_dim: 967
use_optimizer_param_groups: True
freeze_playing_parts: True
BLACKJACK